In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv("../.env")

engine = create_engine(os.getenv("DATABASE_URL"))

def sql_query(query):
    return pd.read_sql(query, engine)

clase_1 = pd.read_csv("../data/clase_1.csv", sep=";")
clase_2 = pd.read_csv("../data/clase_2.csv", sep=";")
clase_3 = pd.read_csv("../data/clase_3.csv", sep=";")
clase_4 = pd.read_csv("../data/clase_4.csv", sep=";")
claustro = pd.read_csv("../data/claustro.csv", sep=";")

clase_1["Vertical"] = "DS"
clase_2["Vertical"] = "DS"
clase_3["Vertical"] = "FS"
clase_4["Vertical"] = "FS"

alumnos = pd.concat([clase_1, clase_2, clase_3, clase_4], ignore_index=True)
alumnos["Fecha"] = pd.to_datetime(alumnos["Fecha_comienzo"], format="%d/%m/%Y").dt.date

if "Fecha_comienzo" in claustro.columns:
    claustro["Fecha"] = pd.to_datetime(claustro["Fecha_comienzo"], format="%d/%m/%Y").dt.date
else:
    fechas_conocidas = alumnos[["Promoción", "Fecha"]].drop_duplicates()

    meses_repetidos = fechas_conocidas[fechas_conocidas["Promoción"].duplicated(keep=False)]
    if len(meses_repetidos) > 0:
        print(meses_repetidos)
        raise ValueError(
            "Hay varias promociones con el mismo mes. "
            "El claustro necesita una columna de fecha para saber a cuál va cada profesor."
        )

    claustro = claustro.merge(fechas_conocidas, left_on="Promocion", right_on="Promoción", how="left")

In [ ]:
df_alumnos = alumnos[["Nombre", "Email"]]
df_alumnos = df_alumnos.rename(columns={"Nombre": "nombre_alumno", "Email": "email"})
df_alumnos = df_alumnos.drop_duplicates(subset="email")


In [6]:
df_alumnos.to_sql("alumnos", con=engine, if_exists="append", index=False)

DatabaseError: Execution failed on sql 'INSERT INTO alumnos (nombre_alumno, email) VALUES (:nombre_alumno, :email)': (psycopg.errors.UniqueViolation) duplicate key value violates unique constraint "alumnos_email_key"
DETAIL:  Key (email)=(Jafet_Casals@gmail.com) already exists.
[SQL: INSERT INTO alumnos (nombre_alumno, email) VALUES (%(nombre_alumno)s::VARCHAR, %(email)s::VARCHAR)]
[parameters: [{'nombre_alumno': 'Jafet Casals', 'email': 'Jafet_Casals@gmail.com'}, {'nombre_alumno': 'Jorge Manzanares', 'email': 'Jorge_Manzanares@gmail.com'}, {'nombre_alumno': 'Onofre Adadia', 'email': 'Onofre_Adadia@gmail.com'}, {'nombre_alumno': 'Merche Prada', 'email': 'Merche_Prada@gmail.com'}, {'nombre_alumno': 'Pilar Abella', 'email': 'Pilar_Abella@gmail.com'}, {'nombre_alumno': 'Leoncio Tena', 'email': 'Leoncio_Tena@gmail.com'}, {'nombre_alumno': 'Odalys Torrijos', 'email': 'Odalys_Torrijos@gmail.com'}, {'nombre_alumno': 'Eduardo Caparrós', 'email': 'Eduardo_Caparrós@gmail.com'}  ... displaying 10

In [5]:
sql_query("SELECT * FROM alumnos LIMIT 5")

,id_alumno,nombre_alumno,email
0,1,Jafet Casals,Jafet_Casals@gmail.com
1,2,Jorge Manzanares,Jorge_Manzanares@gmail.com
2,3,Onofre Adadia,Onofre_Adadia@gmail.com
3,4,Merche Prada,Merche_Prada@gmail.com
4,5,Pilar Abella,Pilar_Abella@gmail.com


In [8]:
df_profesores = claustro[["Nombre"]]
df_profesores = df_profesores.rename(columns={"Nombre": "nombre_profesor"})
df_profesores = df_profesores.drop_duplicates(subset="nombre_profesor")


In [9]:
df_profesores.to_sql("profesores", con=engine, if_exists="append", index=False)

PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/21/8s2b)